# Book Market Analysis: Web Scraping + Cleaning + Analysis + Plotly

**Goal:** scrape book data from a website, clean it, answer business questions with pandas, and show the answers with charts.

**Business questions**
1. Which categories have the most books?
2. Which categories are the most expensive and the cheapest (on average)?
3. Do higher-rated books cost more?
4. How are prices spread out (budget vs mid vs premium)?

**Tools:** Selenium + BeautifulSoup (scraping), pandas (cleaning and analysis), Plotly Express (charts)

**About the data:** [books.toscrape.com](https://books.toscrape.com) is a practice website built for learning web scraping (scraping it is allowed). The prices and ratings are not real market data, so the *method* is what matters here. The same steps work on a real website.

**How to run:** install the libraries with `pip install -r requirements.txt` (you also need Google Chrome installed), then run the cells from top to bottom.

In [ ]:
import os
import time
import pandas as pd
import nbformat
import plotly.io as pio
pio.renderers.default = "browser"
import plotly.express as px
from bs4 import BeautifulSoup
from selenium import webdriver
from urllib.parse import urljoin

## Step 1: Web scraping

**Plan**
1. Open the home page and collect the link of every book category (about 50).
2. Open each category, read every book on the page, then click "next" until there are no more pages.
3. For each book, save: title, category, price, rating and availability.

First, a function that reads **one page** (BeautifulSoup does the reading). Each book on the page sits inside an `<article class="product_pod">` tag.

In [91]:
BASE_URL = "https://books.toscrape.com/"

def parse_books(soup, category):
    """Read every book on one page and return a list of dictionaries (one per book)."""
    books = []
    for card in soup.select("article.product_pod"):
        books.append({
            "title": card.h3.a["title"],                                  # full title is in the title attribute
            "category": category,
            "price": card.select_one("p.price_color").text,               # e.g. "£51.77"
            "rating": card.select_one("p.star-rating")["class"][1],       # e.g. "Three" (a word, not a number)
            "availability": card.select_one("p.availability").text.strip()
        })
    return books

Now the function that uses **Selenium** to open the pages. Selenium opens Chrome and loads each page, then we hand the page's HTML (`driver.page_source`) to BeautifulSoup.

The `while url:` loop keeps going to the next page until a category has no "next" button.

In [92]:
def scrape_all_books():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")   # runs Chrome without a visible window; delete this line to watch it work
    driver = webdriver.Chrome(options=options)
    all_books = []

    try:
        # 1) Home page -> get the list of categories (name + link)
        driver.get(BASE_URL)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        category_links = soup.select("div.side_categories ul li ul li a")
        categories = [(a.text.strip(), urljoin(BASE_URL, a["href"])) for a in category_links]
        print(f"Found {len(categories)} categories")

        # 2) Visit every category, page by page
        for name, url in categories:
            while url:
                driver.get(url)
                time.sleep(0.5)                                   # small pause to be polite to the website
                soup = BeautifulSoup(driver.page_source, "html.parser")
                all_books.extend(parse_books(soup, name))
                next_button = soup.select_one("li.next a")        # is there a "next" page?
                url = urljoin(url, next_button["href"]) if next_button else None
            print(f"{name}: done | books collected so far = {len(all_books)}")
    finally:
        driver.quit()                                             # always close the browser, even if an error happens

    return pd.DataFrame(all_books)

Run the scraper (takes a few minutes). The result is saved to `books_raw.csv`, so next time the notebook loads the saved file instead of scraping again. Delete the file if you want fresh data.

In [93]:
if os.path.exists("books_raw.csv"):
    df_raw = pd.read_csv("books_raw.csv")
    print("Loaded saved data (delete books_raw.csv to scrape again)")
else:
    df_raw = scrape_all_books()
    df_raw.to_csv("books_raw.csv", index=False)
    print("Scraping finished and saved to books_raw.csv")

df_raw.head()

Loaded saved data (delete books_raw.csv to scrape again)


,title,category,price,rating,availability
0,It's Only the Himalayas,Travel,£45.17,Two,In stock
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,£49.43,Four,In stock
2,See America: A Celebration of Our National Par...,Travel,£48.87,Three,In stock
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,£36.94,Two,In stock
4,Under the Tuscan Sun,Travel,£37.33,Three,In stock


## Step 2: Data cleaning

Always keep the raw data untouched and clean a **copy**. First, look at what we have.

In [94]:
df = df_raw.copy()
print("Rows, columns:", df.shape)
df.info()

Rows, columns: (1000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   title         1000 non-null   str  
 1   category      1000 non-null   str  
 2   price         1000 non-null   str  
 3   rating        1000 non-null   str  
 4   availability  1000 non-null   str  
dtypes: str(5)
memory usage: 39.2 KB


**Check for missing values and duplicates**

In [95]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Rows after removing duplicates:", len(df))

Missing values per column:
title           0
category        0
price           0
rating          0
availability    0
dtype: int64

Duplicate rows: 0
Rows after removing duplicates: 1000


**Problem 1: `price` is text** (like `£51.77`), so we cannot calculate with it. We remove everything except digits and the decimal point, then convert to a number. (This also fixes the odd `Â` character that sometimes appears before `£`.)

In [96]:
df["price"] = df["price"].str.replace("[^0-9.]", "", regex=True).astype(float)
df["price"].describe()

count    1000.00000
mean       35.07035
std        14.44669
min        10.00000
25%        22.10750
50%        35.98000
75%        47.45750
max        59.99000
Name: price, dtype: float64

**Problem 2: `rating` is a word** (`One`, `Two`, ...). Convert it to a number 1 to 5 so we can average it and compare it.

In [97]:
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["rating"] = df["rating"].map(rating_map)

print("Ratings that could not be converted:", df["rating"].isnull().sum())
df["rating"].value_counts().sort_index()

Ratings that could not be converted: 0


rating
1    226
2    196
3    203
4    179
5    196
Name: count, dtype: int64

**Problem 3: tidy the text columns and check `availability`.** If a column has the same value in every row, it tells us nothing, so we drop it.

In [98]:
df["title"] = df["title"].str.strip()
df["category"] = df["category"].str.strip()

print(df["availability"].value_counts())
if df["availability"].nunique() == 1:
    df = df.drop(columns="availability")
    print("\nDropped 'availability': every book has the same value")

availability
In stock    1000
Name: count, dtype: int64

Dropped 'availability': every book has the same value


**New column: `price_band`.** Group prices into Budget, Mid and Premium to make analysis easier.

In [99]:
df["price_band"] = pd.cut(
    df["price"],
    bins=[0, 20, 40, float("inf")],
    labels=["Budget (under 20)", "Mid (20 to 40)", "Premium (over 40)"]
)
df.head()

,title,category,price,rating,price_band
0,It's Only the Himalayas,Travel,45.17,2,Premium (over 40)
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,49.43,4,Premium (over 40)
2,See America: A Celebration of Our National Par...,Travel,48.87,3,Premium (over 40)
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,2,Mid (20 to 40)
4,Under the Tuscan Sun,Travel,37.33,3,Mid (20 to 40)


**Final checks, then save the clean data.** `assert` stops the notebook with an error if something is wrong, which is a quick way to prove the cleaning worked.

In [100]:
assert df["price"].notna().all(), "some prices are missing"
assert df["price"].gt(0).all(), "some prices are zero or negative"
assert df["rating"].between(1, 5).all(), "some ratings are outside 1-5"
assert df.duplicated().sum() == 0, "duplicates still present"

df.to_csv("books_clean.csv", index=False)
print("All checks passed. Clean rows:", len(df))

All checks passed. Clean rows: 1000


## Step 3: Analysis (pandas)

### Q1. Which categories have the most books?
We build one summary table per category: number of books, average price and average rating.

In [101]:
category_summary = (
    df.groupby("category")
      .agg(books=("title", "count"), avg_price=("price", "mean"), avg_rating=("rating", "mean"))
      .round(2)
      .sort_values("books", ascending=False)
)
category_summary.head(10)

,books,avg_price,avg_rating
category,,,
Default,152,34.39,2.84
Nonfiction,110,34.26,2.88
Sequential Art,75,34.57,2.97
Add a comment,67,35.80,2.76
Fiction,65,36.07,3.18
Young Adult,54,35.45,3.30
Fantasy,48,39.59,3.08
Romance,35,33.93,2.63
Mystery,32,31.72,2.94


### Q2. Which categories are the most and least expensive?
Categories with only 1 or 2 books give unreliable averages, so we only compare categories with **at least 10 books**.

In [102]:
big_categories = category_summary[category_summary["books"] >= 10]

print("Most expensive categories (avg price):")
print(big_categories.sort_values("avg_price", ascending=False).head(5)[["books", "avg_price"]])

print("\nCheapest categories (avg price):")
print(big_categories.sort_values("avg_price").head(5)[["books", "avg_price"]])

Most expensive categories (avg price):
                books  avg_price
category                        
Travel             11      39.79
Fantasy            48      39.59
History            18      37.29
Womens Fiction     17      36.79
Classics           19      36.55

Cheapest categories (avg price):
                books  avg_price
category                        
Food and Drink     30      31.41
Thriller           11      31.43
Mystery            32      31.72
Business           12      32.46
Childrens          29      32.64


### Q3. Do higher-rated books cost more?
Compare the average price for each rating, then calculate the **correlation** (a number from -1 to +1; close to 0 means almost no relationship).

In [103]:
print(df.groupby("rating")["price"].agg(["count", "mean", "median"]).round(2))

correlation = df["price"].corr(df["rating"])
print("\nCorrelation between price and rating:", round(correlation, 3))

        count   mean  median
rating                      
1         226  34.56   34.77
2         196  34.81   36.22
3         203  34.69   33.78
4         179  36.09   37.80
5         196  35.37   36.90

Correlation between price and rating: 0.028


### Q4. How are prices spread across budget, mid and premium?

In [104]:
band_counts = df["price_band"].value_counts().sort_index()
band_percent = (band_counts / len(df) * 100).round(1)
pd.DataFrame({"books": band_counts, "percent": band_percent})

,books,percent
price_band,,
Budget (under 20),196,19.6
Mid (20 to 40),401,40.1
Premium (over 40),403,40.3


### Optional: is the price difference real? (t-test)
*Only do this if you have learned hypothesis testing.* Question: do **5-star** books have a different average price from **1 and 2-star** books?
Rule of thumb: if the **p-value is below 0.05**, the difference is unlikely to be just chance.

In [105]:
from scipy import stats

five_star = df[df["rating"] == 5]["price"]
low_star = df[df["rating"] <= 2]["price"]

t_stat, p_value = stats.ttest_ind(five_star, low_star, equal_var=False)
print(f"5-star average price : {five_star.mean():.2f}  (n={len(five_star)})")
print(f"1-2 star average price: {low_star.mean():.2f}  (n={len(low_star)})")
print(f"p-value = {p_value:.3f} ->", "significant difference" if p_value < 0.05 else "no significant difference")

5-star average price : 35.37  (n=196)
1-2 star average price: 34.68  (n=422)
p-value = 0.582 -> no significant difference


## Step 4: Charts (Plotly Express)
Each chart answers one of the questions above. Hover over the charts to see exact values.

In [106]:
# Chart 1: categories with the most books
top_cat = category_summary.head(10).reset_index()
fig = px.bar(top_cat, x="books", y="category", orientation="h", text="books",
             title="Top 10 categories by number of books", template="plotly_white")
fig.update_layout(yaxis=dict(autorange="reversed"), xaxis_title="Number of books", yaxis_title="")
fig.show()

In [107]:
# Chart 2: average price by category (categories with at least 10 books)
price_cat = big_categories.sort_values("avg_price", ascending=False).reset_index()
fig = px.bar(price_cat, x="category", y="avg_price", text="avg_price",
             title="Average book price by category (categories with 10+ books)", template="plotly_white")
fig.update_layout(xaxis_title="", yaxis_title="Average price")
fig.show()

In [108]:
# Chart 3: how prices are distributed
fig = px.histogram(df, x="price", nbins=20, title="Distribution of book prices", template="plotly_white")
fig.update_layout(xaxis_title="Price", yaxis_title="Number of books")
fig.show()

In [109]:
# Chart 4: price for each rating (answers Q3)
fig = px.box(df, x="rating", y="price", title="Book price by star rating", template="plotly_white")
fig.update_layout(xaxis_title="Star rating", yaxis_title="Price")
fig.show()

In [110]:
# Chart 5: each bubble is a category; bigger bubble = more books
bubble = big_categories.reset_index()
fig = px.scatter(bubble, x="avg_price", y="avg_rating", size="books", hover_name="category",
                 title="Categories: average price vs average rating (bubble size = number of books)",
                 template="plotly_white")
fig.update_layout(xaxis_title="Average price", yaxis_title="Average rating")
fig.show()

In [111]:
# Chart 6: share of books in each price band
fig = px.pie(values=band_counts.values, names=band_counts.index.astype(str),
             title="Share of books by price band", template="plotly_white")
fig.show()

## Step 5: Insights (write these yourself from your own results)

Look at your tables and charts, then write 3 to 4 findings **with numbers**. Use this format: *what you found, the number, what it means*.

Examples of the style (replace with your real results):
- "The `[category]` category has the most books (`[N]`, `[X]`% of the catalogue)."
- "`[Category]` is the most expensive on average (`[price]`) and `[category]` is the cheapest (`[price]`)."
- "The correlation between price and rating is `[r]`, so a higher rating does / does not go with a higher price."
- "`[X]`% of books are in the Mid price band."

**Your findings:**
1.
2.
3.
4.

## Limitations
- This is a practice website, so prices and ratings are not real market data. Do not present these findings as real-world facts.
- Only 5 fields were collected per book. A real project would also collect fields like publisher, date or number of reviews.
- Averages for small categories can be misleading, which is why we filtered to categories with 10+ books.

## Next steps
- Repeat the same pipeline on a real website that allows scraping (check its `robots.txt` and terms first), such as a product listing or sports-stats page.
- Build the same charts in Tableau or Power BI to show a second tool.

In [112]:
import sys
print(sys.executable)
!{sys.executable} -m pip install nbformat ipython

c:\Users\Himanshu\AppData\Local\Programs\Python\Python314\python.exe


In [113]:
import nbformat
print(nbformat.__version__)

5.11.1


In [114]:
import plotly.io as pio
pio.renderers.default = "browser"